# 📊 Sunum Grafikleri Üretici — Slide 7, 8, 9

Bu notebook, **tam olarak sunuma lazım olan grafikleri** üretir ve `Sunum_Grafikleri/` klasörüne kaydeder.

**Üretilen dosyalar:**

| Dosya | Slide | Açıklama |
|-------|-------|----------|
| `slide7_channel_comp_accuracy_mcldnn.png` | Slide 7 (sol) | MCLDNN: AWGN + 3 fading kanal accuracy |
| `slide7_channel_comp_accuracy_petcgdnn.png` | Slide 7 (sağ) | PET-CGDNN: aynı |
| `slide8_model_comp_f1_rayleigh.png` | Slide 8 (sol) | MCLDNN vs PET-CGDNN F1 on Rayleigh |
| `slide8_model_comp_mcc_rayleigh.png` | Slide 8 (sağ) | MCLDNN vs PET-CGDNN MCC on Rayleigh |
| `slide9_confusion_matrix_comparison.png` | Slide 9 | AWGN vs Rayleigh confusion matrix (SNR=+18dB) |

## 1. Ortam Kurulumu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, pickle, glob
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'figure.facecolor': 'white'})

PROJECT_DIR = '/content/drive/MyDrive/AMR-Project'

# Grafiklerin kaydedileceği dizin
SAVE_DIR = os.path.join(PROJECT_DIR, 'Sunum_Grafikleri')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'📂 Proje: {PROJECT_DIR}')
print(f'💾 Çıktı: {SAVE_DIR}')

## 2. Veri Yükleme

In [ ]:
def find_pkl(model, channel, metric='acc.pkl'):
    """Birden fazla olası konumda pkl dosyası arar."""
    tag = f'{model}_{channel}'
    paths = [
        os.path.join(PROJECT_DIR, 'fine_tuning_results', tag, 'predictions', metric),
        os.path.join(PROJECT_DIR, 'fine_tuning_results', tag, metric),
        os.path.join(PROJECT_DIR, 'results', tag, 'predictions', metric),
        os.path.join(PROJECT_DIR, 'results', tag, metric),
        os.path.join(PROJECT_DIR, 'results', model, channel, 'predictions', metric),
    ]
    for p in paths:
        if os.path.exists(p):
            return pickle.load(open(p, 'rb'))
    return None

# --- Tüm verileri yükle ---
models = ['mcldnn', 'petcgdnn']
channels = ['awgn', 'rayleigh', 'rician_k3', 'rician_k10']
metrics = ['acc.pkl', 'f1_macro_scores.pkl', 'mcc_metric_scores.pkl']
metric_keys = ['acc', 'f1', 'mcc']

data = {}  # data[model][channel][metric_key] = dict(snr->value)
for m in models:
    data[m] = {}
    for ch in channels:
        data[m][ch] = {}
        for mk, mf in zip(metric_keys, metrics):
            data[m][ch][mk] = find_pkl(m, ch, mf)

# Durum raporu
print('📋 Veri Durumu:\n')
print(f'{"":>12} | {"AWGN":>8} | {"Rayleigh":>8} | {"Ric.K3":>8} | {"Ric.K10":>8}')
print('-' * 58)
for m in models:
    for mk in metric_keys:
        row = f'{m}/{mk:>3}'
        vals = []
        for ch in channels:
            status = '✅' if data[m][ch][mk] is not None else '❌'
            vals.append(f'{status:>8}')
        print(f'{row:>12} | {"  |  ".join(vals)}')

---
## 3. SLIDE 7 — Channel Impact on Accuracy

Her model için: AWGN + Rayleigh + Rician K=3 + Rician K=10 accuracy eğrileri.

**Çıktı:** `slide7_channel_comp_accuracy_mcldnn.png`, `slide7_channel_comp_accuracy_petcgdnn.png`

In [ ]:
ch_colors = {'awgn': '#4CAF50', 'rayleigh': '#E63946', 'rician_k3': '#FF9800', 'rician_k10': '#2196F3'}
ch_labels = {'awgn': 'AWGN (Baseline)', 'rayleigh': 'Rayleigh', 'rician_k3': 'Rician K=3', 'rician_k10': 'Rician K=10'}
ch_markers = {'awgn': 'o', 'rayleigh': 's', 'rician_k3': '^', 'rician_k10': 'D'}
model_titles = {'mcldnn': 'MCLDNN', 'petcgdnn': 'PET-CGDNN'}

for m in models:
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    plotted = False
    for ch in channels:
        acc = data[m][ch]['acc']
        if acc is None:
            print(f'⚠️ {m}/{ch} accuracy verisi bulunamadı, atlanıyor.')
            continue
        snrs = sorted(acc.keys())
        vals = [acc[s] * 100 for s in snrs]
        ax.plot(snrs, vals, marker=ch_markers[ch], linewidth=2.5, markersize=6,
                label=ch_labels[ch], color=ch_colors[ch])
        plotted = True
    if not plotted:
        plt.close()
        continue
    ax.set_xlabel('Signal to Noise Ratio (dB)', fontsize=13)
    ax.set_ylabel('Classification Accuracy (%)', fontsize=13)
    ax.set_title(f'{model_titles[m]} — Channel Impact on Classification Accuracy', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim([0, 105])
    plt.tight_layout()
    fname = f'slide7_channel_comp_accuracy_{m}.png'
    fig.savefig(os.path.join(SAVE_DIR, fname), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ Kaydedildi: {fname}\n')

---
## 4. SLIDE 8 — Model Comparison (F1 & MCC) on Rayleigh

En zor senaryo olan Rayleigh kanalında MCLDNN vs PET-CGDNN karşılaştırması.

**Çıktı:** `slide8_model_comp_f1_rayleigh.png`, `slide8_model_comp_mcc_rayleigh.png`

In [ ]:
m_colors = {'mcldnn': '#E63946', 'petcgdnn': '#1D3557'}
m_markers = {'mcldnn': 'o', 'petcgdnn': 's'}

# --- F1 Score ---
mcl_f1 = data['mcldnn']['rayleigh']['f1']
pet_f1 = data['petcgdnn']['rayleigh']['f1']

if mcl_f1 and pet_f1:
    snrs = sorted(mcl_f1.keys())
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(snrs, [mcl_f1[s] for s in snrs], 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    ax.plot(snrs, [pet_f1[s] for s in snrs], 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    ax.set_xlabel('SNR (dB)', fontsize=13)
    ax.set_ylabel('F1 Score (Macro)', fontsize=13)
    ax.set_title('Model F1 Score Comparison on Rayleigh Fading Channel', fontsize=14, fontweight='bold')
    ax.legend(fontsize=12, loc='lower right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide8_model_comp_f1_rayleigh.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ Kaydedildi: slide8_model_comp_f1_rayleigh.png')
else:
    print('❌ F1 verisi eksik! Rayleigh fine-tuning sonuçlarını kontrol edin.')

# --- MCC ---
mcl_mcc = data['mcldnn']['rayleigh']['mcc']
pet_mcc = data['petcgdnn']['rayleigh']['mcc']

if mcl_mcc and pet_mcc:
    snrs = sorted(mcl_mcc.keys())
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(snrs, [mcl_mcc[s] for s in snrs], 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    ax.plot(snrs, [pet_mcc[s] for s in snrs], 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    ax.set_xlabel('SNR (dB)', fontsize=13)
    ax.set_ylabel('MCC', fontsize=13)
    ax.set_title('Model MCC Comparison on Rayleigh Fading Channel', fontsize=14, fontweight='bold')
    ax.legend(fontsize=12, loc='lower right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim([-0.1, 1.05])
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide8_model_comp_mcc_rayleigh.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ Kaydedildi: slide8_model_comp_mcc_rayleigh.png')
else:
    print('❌ MCC verisi eksik! Rayleigh fine-tuning sonuçlarını kontrol edin.')

---
## 5. SLIDE 8 (alt) — Performans Özet Tablosu

Tüm model × kanal kombinasyonları için istatistiksel özet.

In [ ]:
print('=' * 80)
print('📊 PERFORMANS ÖZET TABLOSU')
print('=' * 80)
print(f'{"Model":>10} | {"Channel":>12} | {"Avg Acc%":>8} | {"Max Acc%":>8} | {"Avg F1":>7} | {"Avg MCC":>8}')
print('-' * 80)
for m in models:
    for ch in channels:
        acc = data[m][ch]['acc']
        f1 = data[m][ch]['f1']
        mcc = data[m][ch]['mcc']
        if acc is None:
            continue
        avg_acc = np.mean(list(acc.values())) * 100
        max_acc = np.max(list(acc.values())) * 100
        avg_f1 = f'{np.mean(list(f1.values())):.4f}' if f1 else 'N/A'
        avg_mcc = f'{np.mean(list(mcc.values())):.4f}' if mcc else 'N/A'
        print(f'{model_titles[m]:>10} | {ch_labels[ch]:>12} | {avg_acc:>7.2f}% | {max_acc:>7.2f}% | {avg_f1:>7} | {avg_mcc:>8}')
print('=' * 80)

---
## 6. SLIDE 9 — Confusion Matrix Karşılaştırması (AWGN vs Rayleigh)

Aynı modelin (MCLDNN) yüksek SNR'de (+18 dB) AWGN ve Rayleigh test verileri üzerindeki confusion matrix'leri.

**Ön koşul:** `amr_all_in_one_core_toolkit.py` Drive'da mevcut olmalı.

**Çıktı:** `slide9_confusion_matrix_comparison.png`

In [ ]:
# Toolkit'i yükle
sys.path.insert(0, PROJECT_DIR)
from amr_all_in_one_core_toolkit import (
    load_data, prepare_data_mcldnn, MCLDNN, calculate_confusion_matrix
)
import tensorflow as tf
print(f'TF: {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Dataset yolları
AWGN_PKL = os.path.join(PROJECT_DIR, 'RML2016.10a_dict.pkl')
RAY_PKL  = os.path.join(PROJECT_DIR, 'RML2016.10a_rayleigh.pkl')

# AWGN-eğitimli MCLDNN ağırlıkları (olası konumlar)
weight_candidates = [
    os.path.join(PROJECT_DIR, 'results', 'mcldnn_awgn', 'best_weights.keras'),
    os.path.join(PROJECT_DIR, 'results', 'mcldnn_awgn', 'best_weights.h5'),
    os.path.join(PROJECT_DIR, 'results', 'mcldnn_awgn', 'best_weights.weights.h5'),
]
WEIGHTS = None
for w in weight_candidates:
    if os.path.exists(w):
        WEIGHTS = w
        break

print(f'AWGN dataset: {"✅" if os.path.exists(AWGN_PKL) else "❌"}')
print(f'Rayleigh dataset: {"✅" if os.path.exists(RAY_PKL) else "❌"}')
print(f'MCLDNN weights: {"✅ " + WEIGHTS if WEIGHTS else "❌ Bulunamadı!"}')

In [ ]:
if WEIGHTS and os.path.exists(AWGN_PKL) and os.path.exists(RAY_PKL):
    # Veri yükle
    (mods, snrs, lbl_a), _, _, (Xt_a, Yt_a), (_, _, tidx_a) = load_data(AWGN_PKL)
    (_, _, lbl_r), _, _, (Xt_r, Yt_r), (_, _, tidx_r) = load_data(RAY_PKL)

    # Model yükle
    tf.keras.backend.clear_session()
    model = MCLDNN(weights=None, classes=len(mods))
    model.load_weights(WEIGHTS)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    print('✅ MCLDNN yüklendi.')

    TARGET_SNR = 18  # Yüksek SNR

    def get_cm_at_snr(X_test, Y_test, lbl, test_idx, target_snr):
        """Belirli SNR'de confusion matrix hesapla."""
        test_snrs = [lbl[i][1] for i in test_idx]
        mask = np.where(np.array(test_snrs) == target_snr)
        X_snr = X_test[mask]
        Y_snr = Y_test[mask]
        inp = prepare_data_mcldnn(X_snr, X_snr, X_snr)
        Y_hat = model.predict(inp['test'], batch_size=400)
        cm, cor, ncor = calculate_confusion_matrix(Y_snr, Y_hat, mods)
        acc = cor / (cor + ncor)
        return cm, acc

    cm_awgn, acc_awgn = get_cm_at_snr(Xt_a, Yt_a, lbl_a, tidx_a, TARGET_SNR)
    cm_ray, acc_ray   = get_cm_at_snr(Xt_r, Yt_r, lbl_r, tidx_r, TARGET_SNR)

    # --- Yan yana confusion matrix çizimi ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=150)

    for ax, cm, title_str in [
        (ax1, cm_awgn, f'AWGN (Acc={acc_awgn*100:.1f}%)'),
        (ax2, cm_ray, f'Rayleigh (Acc={acc_ray*100:.1f}%)')
    ]:
        im = ax.imshow(cm * 100, interpolation='nearest', cmap='Blues', vmin=0, vmax=100)
        ax.set_title(title_str, fontsize=13, fontweight='bold')
        ticks = np.arange(len(mods))
        ax.set_xticks(ticks)
        ax.set_xticklabels(mods, rotation=90, fontsize=9)
        ax.set_yticks(ticks)
        ax.set_yticklabels(mods, fontsize=9)
        for i in range(len(mods)):
            for j in range(len(mods)):
                val = int(np.around(cm[i, j] * 100))
                color = 'darkorange' if i == j else 'black'
                fs = 7 if val == 100 else 8
                ax.text(j, i, val, ha='center', va='center', fontsize=fs, color=color)
        fig.colorbar(im, ax=ax, shrink=0.8)

    fig.suptitle(f'MCLDNN Confusion Matrix — SNR = +{TARGET_SNR} dB', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide9_confusion_matrix_comparison.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ Kaydedildi: slide9_confusion_matrix_comparison.png')
else:
    print('❌ Gerekli dosyalar eksik! Confusion matrix oluşturulamıyor.')
    print('   Gerekli: AWGN dataset + Rayleigh dataset + MCLDNN AWGN weights')

---
## 7. Üretilen Dosyaların Özeti

In [ ]:
print('\n' + '=' * 60)
print('📁 ÜRETİLEN DOSYALAR')
print('=' * 60)
print(f'Dizin: {SAVE_DIR}\n')

expected = [
    'slide7_channel_comp_accuracy_mcldnn.png',
    'slide7_channel_comp_accuracy_petcgdnn.png',
    'slide8_model_comp_f1_rayleigh.png',
    'slide8_model_comp_mcc_rayleigh.png',
    'slide9_confusion_matrix_comparison.png',
]

for f in expected:
    path = os.path.join(SAVE_DIR, f)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  ✅ {f} ({size_kb:.0f} KB)')
    else:
        print(f'  ❌ {f} — oluşturulamadı')

print(f'\n📌 Bu dosyaları sunuma eklemek için Drive > AMR-Project > Sunum_Grafikleri klasörüne bakın.')